[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/practical-biopython-course/blob/main/notebooks/02_seq_objects.ipynb)

# 02 · The Seq Object: Transcription, Translation, and GC Content

*Practical Biopython Course · Hamza Hentabli, Ph.D. · ~45 min*

`Bio.Seq.Seq` is a string that *knows biology*: it can complement, transcribe, and translate.
We work with the real human **EGFR mRNA** (RefSeq `NM_005228.5`, downloaded from NCBI and cached in
`data/entrez_cache/`).

**Learning objectives.** By the end of this notebook you will be able to:

- create `Seq` objects and use string-like operations on them
- compute the complement, reverse complement, and GC content
- transcribe DNA to RNA and translate it to protein (including stop codons and genetic codes)
- translate all six reading frames and find an open reading frame

In [ ]:
# Setup: installs dependencies on Google Colab and locates the course data.
# Running locally? Install requirements.txt once and this cell does nothing extra.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'biopython'], check=True)
    if not os.path.isdir("practical-biopython-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/practical-biopython-course.git"], check=True)
    DATA_DIR = "practical-biopython-course/data"
else:
    DATA_DIR = "../data" if os.path.isdir("../data") else "data"

print("Running on Colab:", IN_COLAB, "| data folder:", DATA_DIR)

## 1. Creating and slicing a Seq

In [ ]:
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction

s = Seq("ATGCGACCCTCCGGGACGGCCGGGGCAGCGCTCCTGGCG")
print(type(s), len(s))
print(s[:9], "|", s[-3:], "|", s.count("G"))
print("upper/lower:", s.lower()[:12])
print("find 'GGG' at index:", s.find("GGG"))

## 2. Complement and reverse complement

DNA is double-stranded and antiparallel. The **reverse complement** is the other strand read 5'→3';
you need it whenever a gene or primer lies on the minus strand.

In [ ]:
print("5'-", s, "-3'  (coding strand)")
print("3'-", s.complement(), "-5'  (complement)")
print("5'-", s.reverse_complement(), "-3'  (reverse complement)")

## 3. GC content

In [ ]:
print(f"GC fraction: {gc_fraction(s):.3f}")

Let us load the whole EGFR mRNA. For now we read the FASTA file as text; Notebook 03 introduces
`SeqIO` properly.

In [ ]:
with open(f"{DATA_DIR}/entrez_cache/NM_005228.fasta") as fh:
    header = fh.readline().strip()
    mrna = Seq("".join(line.strip() for line in fh))
print(header)
print(len(mrna), "nt | GC fraction:", round(gc_fraction(mrna), 3))

In [ ]:
import matplotlib.pyplot as plt

window = 200
gc_profile = [gc_fraction(mrna[i:i + window]) for i in range(0, len(mrna) - window, 25)]
plt.figure(figsize=(9, 3))
plt.plot(range(0, len(mrna) - window, 25), gc_profile)
plt.axvspan(261, 3894, color="tab:orange", alpha=0.15, label="coding sequence (CDS)")
plt.xlabel("position in mRNA (nt)"); plt.ylabel(f"GC fraction ({window}-nt window)")
plt.legend(); plt.tight_layout(); plt.show()

The CDS coordinates (261–3894, zero-based) come from the GenBank record; Notebook 04 shows how to
read them programmatically. The 5' region is GC-rich, a common feature of promoter-proximal sequence.

## 4. Transcription and translation

Biopython follows the convention that `transcribe()` takes the **coding strand** and simply
replaces T with U. `translate()` uses the standard genetic code (NCBI table 1) by default; stops are `*`.

In [ ]:
cds = mrna[261:3894]
print("CDS length:", len(cds), "nt =", len(cds) // 3, "codons")
rna = cds.transcribe()
print("mRNA start:", rna[:30])
protein = cds.translate()
print("protein start:", protein[:30], "| ends with:", protein[-5:])

In [ ]:
protein_to_stop = cds.translate(to_stop=True)
print(len(protein), "with stop vs", len(protein_to_stop), "without")
print("starts with Met:", protein_to_stop[0] == "M")

Compare with the UniProt sequence of EGFR (P00533). If our CDS boundaries are right, they match exactly.

In [ ]:
with open(f"{DATA_DIR}/sequences/P00533_EGFR_human.fasta") as fh:
    fh.readline()
    uniprot_seq = "".join(line.strip() for line in fh)
print("identical to UniProt P00533:", str(protein_to_stop) == uniprot_seq)

### Genetic codes

Mitochondria and some organisms use different codes. `TGA` is a stop in the standard code but
tryptophan in the vertebrate mitochondrial code (table 2).

In [ ]:
codon = Seq("ATGTGAAGA")
print("standard (table 1):", codon.translate(table=1))
print("vertebrate mitochondrial (table 2):", codon.translate(table=2))

## 5. Six-frame translation and open reading frames

Given an unknown piece of DNA, you do not know the strand or frame. Translate all six and look for the
longest stretch without a stop codon.

In [ ]:
fragment = mrna[:1200]

def six_frames(seq):
    frames = {}
    for strand, s_ in [("+", seq), ("-", seq.reverse_complement())]:
        for f in range(3):
            sub = s_[f:]
            sub = sub[: len(sub) - len(sub) % 3]
            frames[f"{strand}{f + 1}"] = sub.translate()
    return frames

for name, prot in six_frames(fragment).items():
    longest = max(str(prot).split("*"), key=len)
    print(f"frame {name}: {prot.count('*'):>3} stops | longest stop-free stretch {len(longest):>3} aa")

Only one frame is stop-free over a long stretch: the true reading frame of EGFR. Real ORF finders
also require a start codon; try that in the exercises.

## 6. Codon usage

In [ ]:
from collections import Counter

codon_counts = Counter(str(cds[i:i + 3]) for i in range(0, len(cds), 3))
leu = {c: n for c, n in codon_counts.items() if Seq(c).translate() == "L"}
print("Leucine codons used in EGFR:", dict(sorted(leu.items(), key=lambda kv: -kv[1])))

Codon preference matters in practice: when you express a human protein in *E. coli* for a structural
or biochemical assay, rare codons can limit yield, which is why genes are often "codon-optimized".

### ✏️ Exercise 1

The famous EGFR resistance mutation **T790M** changes threonine 790 to methionine. Find the codon for
residue 790 in `cds` (residue *n* occupies `cds[3*(n-1):3*n]`), translate it, and show which single base
change turns it into a methionine codon (`ATG`).

*Hint:* Python is zero-based, so residue 790 starts at index 3 × 789.

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
codon_790 = cds[3 * 789: 3 * 790]
print("codon 790:", codon_790, "->", codon_790.translate())
print("mutant ATG ->", Seq("ATG").translate(), "| changed position:",
      [i for i, (a, b) in enumerate(zip(str(codon_790), "ATG")) if a != b])
```

</details>

### ✏️ Exercise 2

Write `longest_orf(seq)` that scans the three forward frames of a DNA `Seq` and returns the longest
protein that **starts with M** and ends at a stop codon. Test it on `fragment`.

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
def longest_orf(seq):
    best = ""
    for f in range(3):
        sub = seq[f:]
        prot = str(sub[: len(sub) - len(sub) % 3].translate())
        for chunk in prot.split("*")[:-1]:          # chunks that are followed by a stop
            if "M" in chunk:
                orf = chunk[chunk.index("M"):]
                best = max(best, orf, key=len)
    return best

orf = longest_orf(mrna)
print(len(orf), "aa | equals EGFR:", orf == uniprot_seq)
```

</details>

### ✏️ Exercise 3

Compute the GC fraction of the 5' UTR (`mrna[:261]`), the CDS, and the 3' UTR (`mrna[3894:]`).
Which region is the most GC-rich?

In [ ]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
for name, region in [("5' UTR", mrna[:261]), ("CDS", cds), ("3' UTR", mrna[3894:])]:
    print(f"{name:7s} {len(region):>5} nt  GC = {gc_fraction(region):.3f}")
```

</details>

## Key takeaways

- `Seq` behaves like a string and adds `complement`, `reverse_complement`, `transcribe`, and `translate`.
- Use `to_stop=True` and the right `table=` when translating.
- A single base change (ACG→ATG) creates the clinically important T790M mutation.

**Next:** [03 · SeqRecord and SeqIO](03_seqrecord_seqio.ipynb).